In [29]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')
print('✅ All imports successful')
import plotly; print(f'Plotly version: {plotly.__version__}')

✅ All imports successful
Plotly version: 6.7.0


In [30]:
df = pd.read_csv('claim_data.csv')
df['Date of Service'] = pd.to_datetime(df['Date of Service'])
df['Month'] = df['Date of Service'].dt.to_period('M')
df['amount_saved'] = df['Billed Amount'] - df['Paid Amount']

print(f'Dataset loaded: {len(df):,} records | {df.shape[1]} columns')
print(f'Date range: {df["Date of Service"].min().strftime("%b %Y")} → {df["Date of Service"].max().strftime("%b %Y")}')
df.head(3)

Dataset loaded: 1,000 records | 17 columns
Date range: May 2024 → Sep 2024


,Claim ID,Provider ID,Patient ID,Date of Service,Billed Amount,Procedure Code,Diagnosis Code,Allowed Amount,Paid Amount,Insurance Type,Claim Status,Reason Code,Follow-up Required,AR Status,Outcome,Month,amount_saved
0,0HO1FSN4AP,126528997,7936697103,2024-08-07,304,99231,A02.1,218,203,Self-Pay,Paid,Incorrect billing information,Yes,Pending,Partially Paid,2024-08,101
1,9U86CI2P5A,6986719948,1547160031,2024-06-21,348,99213,A16.5,216,206,Medicare,Paid,Pre-existing condition,Yes,Open,Denied,2024-06,142
2,1QEU1AIDAU,1355108115,2611585318,2024-07-04,235,99213,A00.1,148,119,Commercial,Under Review,Duplicate claim,No,Denied,Denied,2024-07,116


In [31]:
FRAUD_INDICATORS = ['Duplicate claim', 'Incorrect billing information']

KPI = {
    'approval_rate' : round((df['Claim Status'] == 'Paid').sum() / len(df) * 100, 1),
    'avg_days'       : None,
    'fraud_rate'     : round(df['Reason Code'].isin(FRAUD_INDICATORS).sum() / len(df) * 100, 1),
    'auto_rate'      : round(((df['Outcome'] == 'Paid') & (df['Claim Status'] == 'Paid')).sum() / len(df) * 100, 1),
    'total_saved'    : int(df['amount_saved'].sum())
}

monthly = df.groupby('Month').agg(
    total    = ('Claim ID', 'count'),
    approved = ('Claim Status', lambda x: (x == 'Paid').sum()),
    fraud    = ('Reason Code', lambda x: x.isin(FRAUD_INDICATORS).sum()),
    saved    = ('amount_saved', 'sum'),
    auto     = ('Claim ID', lambda x: (
                  (df.loc[x.index, 'Outcome'] == 'Paid') &
                  (df.loc[x.index, 'Claim Status'] == 'Paid')
                ).sum())
).reset_index()

monthly['approval_rate'] = round(monthly['approved'] / monthly['total'] * 100, 1)
monthly['fraud_rate']    = round(monthly['fraud']    / monthly['total'] * 100, 1)
monthly['auto_rate']     = round(monthly['auto']     / monthly['total'] * 100, 1)
monthly['month_str']     = monthly['Month'].astype(str)

print('── KPI SUMMARY ─────────────────────────────────')
print(f'  Approval Rate       : {KPI["approval_rate"]}%')
print(f'  Avg Days to Decision: BLOCKED (FS-06 — no Decision Date)')
print(f'  Fraud Flag Rate     : {KPI["fraud_rate"]}%')
print(f'  Auto-Decision Rate  : {KPI["auto_rate"]}% (proxy)')
print(f'  £ Saved             : £{KPI["total_saved"]:,}')
print('─────────────────────────────────────────────────')

── KPI SUMMARY ─────────────────────────────────
  Approval Rate       : 33.4%
  Avg Days to Decision: BLOCKED (FS-06 — no Decision Date)
  Fraud Flag Rate     : 25.5%
  Auto-Decision Rate  : 11.6% (proxy)
  £ Saved             : £96,437
─────────────────────────────────────────────────


In [32]:
NHS_BLUE   = '#003087'
NHS_GREEN  = '#009639'
NHS_RED    = '#DA291C'
NHS_ORANGE = '#ED8B00'
NHS_DARK   = '#231f20'

fig_cards = make_subplots(
    rows=1, cols=5,
    specs=[[{'type': 'indicator'}] * 5],
    subplot_titles=[
        'Approval Rate', 'Avg Days to Decision',
        'Fraud Flag Rate', 'Auto-Decision Rate', '£ Saved'
    ]
)

cards = [
    (KPI['approval_rate'], NHS_GREEN,  False, False, '',  '%'),
    (0,                    NHS_ORANGE, True,  False, '',  ' days'),
    (KPI['fraud_rate'],    NHS_RED,    False, False, '',  '%'),
    (KPI['auto_rate'],     NHS_BLUE,   False, True,  '',  '%'),
    (KPI['total_saved'],   NHS_DARK,   False, False, '£', ''),
]
labels = ['Approval Rate', 'Avg Days to Decision', 'Fraud Flag Rate', 'Auto-Decision Rate', '£ Saved']

for i, (val, color, blocked, proxy, prefix, suffix) in enumerate(cards):
    fig_cards.add_trace(go.Indicator(
        mode='number',
        value=None if blocked else val,
        number=dict(
            prefix=prefix, suffix=suffix,
            font=dict(size=40, color=color, family='Arial Black')
        ),
        title=dict(
            text=(
                f'<b>{labels[i]}</b>'
                + ('<br><span style="font-size:10px;color:#e65100">⚠ Blocked — FS-06</span>' if blocked else '')
                + ('<br><span style="font-size:10px;color:#f57f17">⚠ Proxy — ML-04</span>' if proxy else '')
            ),
            font=dict(size=13, color='#333', family='Arial')
        ),
        domain=dict(row=0, column=i)
    ), row=1, col=i+1)

fig_cards.update_layout(
    title=dict(
        text='<b>NHS Claims Analytics Dashboard</b> — KPI Overview<br><sup>AI Insurance Claim Automation | DS-03 W2 Prototype | claim_data.csv (1,000 records)</sup>',
        font=dict(size=18, color=NHS_BLUE, family='Arial'),
        x=0.5, xanchor='center'
    ),
    height=220, paper_bgcolor='white', plot_bgcolor='white',
    margin=dict(t=80, b=10, l=20, r=20)
)

fig_cards.show()
print('✅ KPI Cards rendered')

✅ KPI Cards rendered


In [33]:
months = monthly['month_str'].tolist()

fig_trends = make_subplots(
    rows=1, cols=3,
    subplot_titles=[
        'Approval Rate % — Monthly Trend',
        'Fraud Flag Rate % — Monthly Trend',
        '£ Saved — Monthly Trend'
    ],
    horizontal_spacing=0.08
)

fig_trends.add_trace(go.Scatter(
    x=months, y=monthly['approval_rate'],
    mode='lines+markers+text', name='Approval Rate %',
    line=dict(color=NHS_GREEN, width=3),
    marker=dict(size=10, color=NHS_GREEN, symbol='circle'),
    text=[f'{v}%' for v in monthly['approval_rate']],
    textposition='top center', textfont=dict(size=11, color=NHS_GREEN),
    fill='tozeroy', fillcolor='rgba(0,150,57,0.08)'
), row=1, col=1)

fig_trends.add_trace(go.Scatter(
    x=months, y=monthly['fraud_rate'],
    mode='lines+markers+text', name='Fraud Flag %',
    line=dict(color=NHS_RED, width=3),
    marker=dict(size=10, color=NHS_RED, symbol='diamond'),
    text=[f'{v}%' for v in monthly['fraud_rate']],
    textposition='top center', textfont=dict(size=11, color=NHS_RED),
    fill='tozeroy', fillcolor='rgba(218,41,28,0.08)'
), row=1, col=2)

fig_trends.add_trace(go.Bar(
    x=months, y=monthly['saved'], name='£ Saved',
    marker_color=NHS_BLUE, opacity=0.85,
    text=[f'£{v:,}' for v in monthly['saved']],
    textposition='outside', textfont=dict(size=11, color=NHS_BLUE)
), row=1, col=3)

fig_trends.update_yaxes(title_text='Rate (%)', row=1, col=1, range=[0, 50])
fig_trends.update_yaxes(title_text='Rate (%)', row=1, col=2, range=[0, 50])
fig_trends.update_yaxes(title_text='£ Amount', row=1, col=3)

fig_trends.update_layout(
    title=dict(
        text='<b>Monthly KPI Trends</b> — May to Sep 2024<br><sup>Approval improving ✅ | Fraud declining ✅ | Savings consistent ✅</sup>',
        font=dict(size=16, color=NHS_BLUE, family='Arial'),
        x=0.5, xanchor='center'
    ),
    height=380, showlegend=False, paper_bgcolor='white', plot_bgcolor='#FAFBFD',
    margin=dict(t=90, b=40, l=60, r=40), font=dict(family='Arial')
)

fig_trends.show()
print('✅ Trend charts rendered')

✅ Trend charts rendered


In [34]:
fig_breakdown = make_subplots(
    rows=1, cols=3,
    specs=[[{'type': 'pie'}, {'type': 'pie'}, {'type': 'bar'}]],
    subplot_titles=['Claim Status Distribution', 'Insurance Type Mix', 'Monthly Claim Volume']
)

status_counts = df['Claim Status'].value_counts()
fig_breakdown.add_trace(go.Pie(
    labels=status_counts.index.tolist(), values=status_counts.values.tolist(),
    hole=0.45, marker=dict(colors=[NHS_GREEN, NHS_RED, NHS_ORANGE]),
    textinfo='label+percent', textfont=dict(size=12, family='Arial'), showlegend=False
), row=1, col=1)

ins_counts = df['Insurance Type'].value_counts()
fig_breakdown.add_trace(go.Pie(
    labels=ins_counts.index.tolist(), values=ins_counts.values.tolist(),
    hole=0.45, marker=dict(colors=[NHS_BLUE, '#005EB8', '#0072CE', '#41B6E6']),
    textinfo='label+percent', textfont=dict(size=12, family='Arial'), showlegend=False
), row=1, col=2)

fig_breakdown.add_trace(go.Bar(
    x=monthly['month_str'], y=monthly['total'],
    marker_color=NHS_BLUE, opacity=0.85,
    text=monthly['total'], textposition='outside',
    textfont=dict(size=12, color=NHS_BLUE), showlegend=False
), row=1, col=3)

fig_breakdown.update_yaxes(title_text='Number of Claims', row=1, col=3)
fig_breakdown.update_layout(
    title=dict(
        text='<b>Claims Distribution & Volume</b><br><sup>Status mix | Insurance type | Monthly throughput</sup>',
        font=dict(size=16, color=NHS_BLUE, family='Arial'), x=0.5, xanchor='center'
    ),
    height=380, paper_bgcolor='white', plot_bgcolor='#FAFBFD',
    margin=dict(t=90, b=40, l=40, r=40), font=dict(family='Arial')
)

fig_breakdown.show()
print('✅ Distribution charts rendered')

✅ Distribution charts rendered


In [35]:
auto_count  = int(((df['Outcome'] == 'Paid') & (df['Claim Status'] == 'Paid')).sum())
human_count = len(df) - auto_count

fig_auto = go.Figure(go.Pie(
    labels=['Auto-Decided', 'Human Review Required'],
    values=[auto_count, human_count],
    hole=0.6,
    marker=dict(colors=[NHS_BLUE, '#E0E8F5']),
    textinfo='label+percent',
    textfont=dict(size=13, family='Arial'),
    showlegend=True
))

fig_auto.update_layout(
    title=dict(
        text=f'<b>Auto-Decision Rate — {KPI["auto_rate"]}%</b> (Proxy)<br>'
             f'<sup>⚠ Using Outcome=Paid AND Status=Paid as proxy. Replace with ML-04 auto_decision_flag when available.</sup>',
        font=dict(size=15, color=NHS_BLUE, family='Arial'),
        x=0.5, xanchor='center'
    ),
    annotations=[dict(
        text=f'<b>{KPI["auto_rate"]}%</b><br>Auto',
        x=0.5, y=0.5, font=dict(size=18, color=NHS_BLUE, family='Arial Black'),
        showarrow=False
    )],
    height=350, paper_bgcolor='white',
    margin=dict(t=100, b=40, l=40, r=40)
)

fig_auto.show()
print('✅ Auto-decision donut rendered')

✅ Auto-decision donut rendered


In [ ]:
html_content = f"""<!DOCTYPE html>
<html>
<head>
    <title>DS-03 | NHS Claims Dashboard | W2 Prototype</title>
    <script src="https://cdn.plot.ly/plotly-latest.min.js"></script>
    <style>
        body {{ font-family: Arial, sans-serif; background: #f5f7fa; margin: 0; padding: 20px; }}
        .header {{ background: #003087; color: white; padding: 20px 30px; border-radius: 8px; margin-bottom: 20px; }}
        .header h1 {{ margin: 0; font-size: 22px; }}
        .header p {{ margin: 5px 0 0; font-size: 13px; opacity: 0.85; }}
        .chart-box {{ background: white; border-radius: 8px; padding: 10px; margin-bottom: 20px;
                      box-shadow: 0 2px 8px rgba(0,0,0,0.08); }}
        .blocker {{ background: #fff3e0; border-left: 4px solid #e65100; padding: 12px 16px;
                    border-radius: 4px; margin-bottom: 20px; font-size: 13px; }}
        .blocker strong {{ color: #e65100; }}
    </style>
</head>
<body>
    <div class="header">
        <h1>AI Insurance Claim Automation Analytics Dashboard — DS-03 W2 Prototype</h1>
        <p>Data Science & Analytics Squad | Week 2 | claim_data.csv (1,000 records)</p>
    </div>
    <div class="blocker">
        <strong>⚠ Active Blockers:</strong>
        KPI 2 (Avg Days to Decision) — waiting on <strong>FS-06</strong> for decision_date field. &nbsp;|
        KPI 4 (Auto-Decision Rate) — proxy logic in use, waiting on <strong>ML-04</strong> auto_decision_flag.
    </div>
    <div class="chart-box">{fig_cards.to_html(full_html=False, include_plotlyjs=False)}</div>
    <div class="chart-box">{fig_trends.to_html(full_html=False, include_plotlyjs=False)}</div>
    <div class="chart-box">{fig_breakdown.to_html(full_html=False, include_plotlyjs=False)}</div>
    <div class="chart-box">{fig_auto.to_html(full_html=False, include_plotlyjs=False)}</div>
</body>
</html>"""

with open('W2_Dashboard_Prototype.html', 'w', encoding='utf-8') as f:
    f.write(html_content)

print('✅ Dashboard exported: W2_Dashboard_Prototype.html')

✅ Dashboard exported: W2_Dashboard_Prototype.html
